# 01A — Telecom sector pack

**Outcome:** translate the native synthetic Telecom files into the small
Pack v0.3 interface consumed by the common adapter.

This notebook owns Telecom meaning: native field names, metric units,
ONT identity and fault-label routing. It does not create model features.
`PACK-CORE` and `PACK-EVAL` are written to physically separate folders.
Topology, service windows and engineering events are not used in this
telemetry-only milestone.


## 1. Setup

Keep this notebook and `week1_core.py` together in
`MyDrive/anomaly_detection/research/week1/`. The default run uses the
complete observable panel. Output directories are immutable, so change
`PACK_RUN_ID` before rebuilding a completed run.


In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    EVAL_SCHEMAS,
    PACK_ENTITY_SCHEMA,
    PACK_METRIC_SCHEMA,
    finalise_pack,
    immutable_directory,
    pack_core_hashes,
    read_json,
    sha256_file,
    source_record,
)

SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT",
    DRIVE_ROOT / "telco_syntetic_data",
))
PACK_RUN_ID = os.getenv("TELECOM_PACK_RUN_ID", "telecom_pack_v0_3")
PACK_ROOT = DRIVE_ROOT / "outputs" / "packs" / "telecom" / PACK_RUN_ID
BATCH_ROWS = int(os.getenv("TELECOM_BATCH_ROWS", "100000"))
ENTITY_IDS = tuple(filter(None, os.getenv("TELECOM_ENTITY_IDS", "").split(",")))
SAMPLE_START = os.getenv("TELECOM_SAMPLE_START") or None
SAMPLE_END = os.getenv("TELECOM_SAMPLE_END") or None
RUN_BUILD = os.getenv("RUN_TELECOM_PACK", "1") == "1"

display(pd.Series({
    "source": str(SOURCE),
    "pack_root": str(PACK_ROOT),
    "entities": ENTITY_IDS or "all",
    "sample_start": SAMPLE_START or "first observation",
    "sample_end": SAMPLE_END or "last observation",
    "batch_rows": BATCH_ROWS,
}, name="value").to_frame())


## 2. Telecom phrasebook

`native_field` is used only while reading the source. The remaining fields
form the authored metric catalogue. Every metric is periodic at 900 seconds.

The FEC ceiling is declared explicitly, so a value at `5_000_000` becomes
`quality_code="clipped"`. No CRC ceiling is asserted without source evidence.


In [ ]:
METRIC_COLUMNS = ["native_field", *PACK_METRIC_SCHEMA]
metric_map = pd.DataFrame([
    ("rx_power_dbm", "rx_power_dbm", "ont", "gauge", "dBm", "periodic", 900, None, None, "none", None),
    ("olt_rx_power_dbm", "olt_rx_power_dbm", "ont", "gauge", "dBm", "periodic", 900, None, None, "none", None),
    ("tx_power_dbm", "tx_power_dbm", "ont", "gauge", "dBm", "periodic", 900, None, None, "none", None),
    ("temperature_c", "temperature_c", "ont", "gauge", "degC", "periodic", 900, None, None, "none", None),
    ("bias_current_ma", "bias_current_ma", "ont", "gauge", "mA", "periodic", 900, 0, None, "none", None),
    ("voltage_v", "voltage_v", "ont", "gauge", "V", "periodic", 900, 0, None, "none", None),
    ("ber", "ber", "ont", "bounded_fraction", "ratio", "periodic", 900, 0, 1, "none", None),
    ("fec_count", "fec_count", "ont", "interval_count", "count", "periodic", 900, 0, None, "right", 5_000_000),
    ("crc_errors", "crc_errors", "ont", "interval_count", "count", "periodic", 900, 0, None, "none", None),
    ("uptime_s", "uptime_s", "ont", "cumulative_counter", "s", "periodic", 900, 0, None, "none", None),
    ("reboot_count", "reboot_count", "ont", "cumulative_counter", "count", "periodic", 900, 0, None, "none", None),
    ("throughput_mbps", "throughput_mbps", "ont", "gauge", "Mbps", "periodic", 900, 0, None, "none", None),
], columns=METRIC_COLUMNS)

assert metric_map["metric_id"].is_unique
display(metric_map)


## 3. Inspect the native source

Only the timestamp, ONT identifier and explicitly mapped measurements are
read into `PACK-CORE`. Extra source columns are ignored. The pack fails if
the observable panel still contains recognisable truth fields.

`tickets.csv`, the fault registry and fault intervals are evaluation-only.
Topology, engineering events and service windows are outside this pack.


In [ ]:
def find_file(root, name, required=True):
    matches = sorted(path for path in Path(root).rglob(name) if path.is_file())
    if len(matches) > 1:
        raise ValueError(f"Ambiguous source file {name}: {matches}")
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"Missing source file: {name}")
    return None


def find_panel(root):
    exact = [
        find_file(root, "reference_dataset.parquet", required=False),
        find_file(root, "reference_dataset.csv", required=False),
    ]
    exact = [path for path in exact if path is not None]
    if len(exact) == 1:
        return exact[0]
    candidates = sorted(Path(root).rglob("reference_dataset*.parquet"))
    candidates += sorted(Path(root).rglob("reference_dataset*.csv"))
    if len(candidates) != 1:
        raise FileNotFoundError(f"Expected one reference dataset, found {candidates}")
    return candidates[0]


def panel_columns(path):
    if path.suffix == ".parquet":
        return pq.ParquetFile(path).schema_arrow.names
    return pd.read_csv(path, nrows=0).columns.tolist()


def iter_panel(path, columns, batch_rows):
    if path.suffix == ".parquet":
        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(batch_size=batch_rows, columns=columns):
            yield batch.to_pandas()
    else:
        yield from pd.read_csv(path, usecols=columns, chunksize=batch_rows)


panel_path = find_panel(SOURCE)
native_columns = panel_columns(panel_path)
lower_columns = {name.lower() for name in native_columns}
truth_tokens = ("fault", "label", "anomaly", "root_cause", "ticket")
truth_columns = sorted(
    name for name in native_columns
    if name.lower().startswith("gt_")
    or name.lower() in {"class", "state"}
    or any(token in name.lower() for token in truth_tokens)
)
available_metrics = metric_map.loc[
    metric_map["native_field"].isin(native_columns)
].copy()

ignored_columns = sorted(
    set(native_columns)
    - {"timestamp_utc", "ont_id"}
    - set(available_metrics["native_field"])
)

inventory = {
    "panel": panel_path.name,
    "panel_columns": len(native_columns),
    "mapped_metrics": len(available_metrics),
    "ignored_observable_columns": ignored_columns,
    "truth_columns_in_observable_panel": truth_columns,
    "fault_registry_available": find_file(SOURCE, "gt_fault_registry.csv", False) is not None,
    "fault_intervals_available": find_file(SOURCE, "fault_entity_intervals.csv", False) is not None,
    "tickets_available": find_file(SOURCE, "tickets.csv", False) is not None,
}
display(pd.Series(inventory, name="value").to_frame())

assert {"timestamp_utc", "ont_id"} <= lower_columns
assert not truth_columns, "Use the observable-only reference dataset."
assert not available_metrics.empty


## 4. Build the Telecom pack

The wide observation files remain efficient. The common adapter performs
the long canonical conversion. The entity registry identifies monitored
ONTs only; its observation bounds are derived later by the common adapter.


In [ ]:
def select_rows(frame, entity_ids=(), start=None, end=None):
    selected = frame.copy()
    if entity_ids:
        selected = selected.loc[selected["ont_id"].astype(str).isin(entity_ids)]
    timestamps = pd.to_datetime(selected["timestamp_utc"], utc=True)
    if start:
        selected = selected.loc[timestamps.ge(pd.to_datetime(start, utc=True))]
        timestamps = pd.to_datetime(selected["timestamp_utc"], utc=True)
    if end:
        selected = selected.loc[timestamps.lt(pd.to_datetime(end, utc=True))]
    return selected.reset_index(drop=True)


def pick_column(frame, *names, default=pd.NA):
    for name in names:
        if name in frame:
            return frame[name]
    return pd.Series(default, index=frame.index)


def telecom_evaluation(root, selected_entities):
    registry = pd.read_csv(find_file(root, "gt_fault_registry.csv"))
    intervals = pd.read_csv(find_file(root, "fault_entity_intervals.csv"))
    intervals = intervals.loc[
        intervals["entity_id"].astype(str).isin(selected_entities)
    ].copy()
    selected_faults = set(intervals["fault_id"].dropna().astype(str))
    registry_ids = pick_column(registry, "gt_fault_id", "fault_id").astype("string")
    registry = registry.loc[registry_ids.isin(selected_faults)].copy()

    fault_events = pd.DataFrame({
        "fault_id": pick_column(registry, "gt_fault_id", "fault_id").astype("string"),
        "fault_type": pick_column(registry, "gt_fault_type", "fault_type").astype("string"),
        "domain_id": pick_column(registry, "target", "domain_id").astype("string"),
        "onset_ts": pick_column(registry, "onset_ts"),
        "observable_ts": pick_column(registry, "first_observable_ts", "observable_ts"),
        "impact_ts": pick_column(registry, "impact_ts"),
        "end_ts": pick_column(registry, "repair_ts", "end_ts"),
        "group_id": pick_column(registry, "group_id").astype("string"),
        "label_source": "synthetic_generator_truth",
        "source_instance_id": pd.NA,
    })
    fault_intervals = pd.DataFrame({
        "fault_id": intervals["fault_id"].astype("string"),
        "entity_id": intervals["entity_id"].astype("string"),
        "start_ts": pick_column(intervals, "active_start_ts", "start_ts"),
        "end_ts": pick_column(intervals, "active_end_ts", "end_ts"),
        "label_source": "synthetic_generator_truth",
        "source_instance_id": pd.NA,
    })

    ticket_path = find_file(root, "tickets.csv", required=False)
    if ticket_path is None:
        tickets = pd.DataFrame(columns=EVAL_SCHEMAS["tickets"])
    else:
        native = pd.read_csv(ticket_path)
        native = native.loc[native["ont_id"].astype(str).isin(selected_entities)]
        tickets = pd.DataFrame({
            "ticket_id": native["ticket_id"].astype("string"),
            "entity_id": native["ont_id"].astype("string"),
            "reported_ts": pick_column(native, "reported_ts"),
            "resolved_ts": pick_column(native, "resolved_ts"),
            "fault_id": pick_column(native, "gt_fault_id", "fault_id").astype("string"),
            "reported_fault_type": pick_column(native, "gt_fault_type", "fault_type").astype("string"),
            "is_no_fault_found": pick_column(native, "gt_is_nff", default=False).astype("boolean"),
            "is_misattributed": pick_column(native, "gt_misattributed", default=False).astype("boolean"),
            "label_source": "synthetic_ticket_stream",
        })

    tables = {
        "fault_events": fault_events,
        "fault_entity_intervals": fault_intervals,
        "tickets": tickets,
    }
    for table_name, frame in tables.items():
        for column in frame:
            if column.endswith("_ts"):
                frame[column] = pd.to_datetime(frame[column], utc=True, errors="coerce")
        tables[table_name] = frame[EVAL_SCHEMAS[table_name]]
    return tables


In [ ]:
def build_telecom_pack(
    source,
    destination,
    *,
    include_evaluation=True,
    entity_ids=(),
    start=None,
    end=None,
    batch_rows=100_000,
):
    source, destination = Path(source), Path(destination)
    panel = find_panel(source)
    catalogue = metric_map.loc[
        metric_map["native_field"].isin(panel_columns(panel))
    ].copy()
    read_columns = ["timestamp_utc", "ont_id", *catalogue["native_field"]]
    rename_metrics = dict(zip(catalogue["native_field"], catalogue["metric_id"]))

    with immutable_directory(destination) as pack:
        core = pack / "PACK-CORE"
        observations = core / "observations"
        observations.mkdir(parents=True)

        observed_entities = set()
        part_number = 0
        for batch in iter_panel(panel, read_columns, batch_rows):
            batch = select_rows(batch, entity_ids, start, end)
            if batch.empty:
                continue
            wide = batch.rename(columns={
                "timestamp_utc": "event_ts",
                "ont_id": "entity_id",
                **rename_metrics,
            })
            wide["event_ts"] = pd.to_datetime(wide["event_ts"], utc=True)
            wide["entity_id"] = wide["entity_id"].astype(str)
            wide = wide[["event_ts", "entity_id", *catalogue["metric_id"]]]
            wide.to_parquet(
                observations / f"part-{part_number:05d}.parquet",
                index=False,
                compression="zstd",
            )
            observed_entities.update(wide["entity_id"].unique())
            part_number += 1

        if not observed_entities:
            raise ValueError("The selected telemetry slice is empty")

        catalogue[PACK_METRIC_SCHEMA].to_parquet(
            core / "metric_catalogue.parquet", index=False
        )
        pd.DataFrame({
            "entity_id": sorted(observed_entities),
            "entity_type": "ont",
        })[PACK_ENTITY_SCHEMA].to_parquet(core / "entity_registry.parquet", index=False)

        evaluation_tables = []
        if include_evaluation:
            evaluation = pack / "PACK-EVAL"
            evaluation.mkdir()
            tables = telecom_evaluation(source, observed_entities)
            evaluation_tables = list(tables)
            for name, frame in tables.items():
                frame.to_parquet(evaluation / f"{name}.parquet", index=False)

        source_files = [source_record(panel, source, "model_input")]
        if include_evaluation:
            for name in ("gt_fault_registry.csv", "fault_entity_intervals.csv", "tickets.csv"):
                path = find_file(source, name, required=False)
                if path is not None:
                    source_files.append(source_record(path, source, "evaluation_only"))

        finalise_pack(
            pack,
            sector="telecom",
            pack_version="0.3.0",
            source_manifest={
                "source_id": "telemetry-synth-4.1.0",
                "source_root": str(source),
                "files": source_files,
                "selection": {
                    "entity_ids": list(entity_ids),
                    "start": start,
                    "end": end,
                },
            },
            evaluation_tables=evaluation_tables,
            notes=[
                "FEC values at 5000000 are source-censored.",
                "CRC censoring is not asserted without source evidence.",
                "Tickets are evaluation-only.",
                "Topology and operational context are not read by Pack v0.3.",
            ],
        )
    return read_json(destination / "pack_manifest.json")


if RUN_BUILD:
    pack_manifest = build_telecom_pack(
        SOURCE,
        PACK_ROOT,
        include_evaluation=True,
        entity_ids=ENTITY_IDS,
        start=SAMPLE_START,
        end=SAMPLE_END,
        batch_rows=BATCH_ROWS,
    )
else:
    pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")

display(pd.Series(pack_manifest["core_row_counts"], name="rows").to_frame())


## 5. Truth-isolation test

This is the primary leakage test because the sector notebook is the only
component that sees native observations and native truth together.

The same observable fixture is translated twice: once with fault and ticket
files present, and once after those files are removed. Logical `PACK-CORE`
hashes must be identical. A deliberately leaky ticket signature must change.


In [ ]:
def write_small_panel(source_panel, destination, rows=500):
    columns = ["timestamp_utc", "ont_id", *available_metrics["native_field"]]
    sample = next(iter_panel(source_panel, columns, rows)).head(rows)
    sample.to_parquet(destination / "reference_dataset.parquet", index=False)
    return sample


def make_isolation_fixture(source, destination, include_evaluation):
    destination.mkdir()
    write_small_panel(find_panel(source), destination)
    if include_evaluation:
        for name in ("gt_fault_registry.csv", "fault_entity_intervals.csv", "tickets.csv"):
            path = find_file(source, name, required=False)
            if path is not None:
                shutil.copy2(path, destination / name)


def deliberately_leaky_signature(source):
    tickets = find_file(source, "tickets.csv", required=False)
    return "missing" if tickets is None else sha256_file(tickets)


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    original_source = temporary / "native_original"
    redacted_source = temporary / "native_redacted"
    make_isolation_fixture(SOURCE, original_source, include_evaluation=True)
    make_isolation_fixture(SOURCE, redacted_source, include_evaluation=False)

    original_pack = temporary / "pack_original"
    redacted_pack = temporary / "pack_redacted"
    build_telecom_pack(original_source, original_pack, include_evaluation=True)
    build_telecom_pack(redacted_source, redacted_pack, include_evaluation=False)

    assert pack_core_hashes(original_pack) == pack_core_hashes(redacted_pack)
    assert deliberately_leaky_signature(original_source) != deliberately_leaky_signature(redacted_source)

print("PASS — PACK-CORE is unchanged after all evaluation files are removed")
print("PASS — the negative control detects the missing ticket file")


## 6. Inspect the result

The tables below are small previews. The manifests contain immutable hashes,
source roles and row counts.


In [ ]:
display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "metric_catalogue.parquet"))
display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "entity_registry.parquet").head())
display(pd.Series(read_json(PACK_ROOT / "pack_manifest.json"), name="value").to_frame())
print("Pack root:", PACK_ROOT)
print("Next: 01B_COMMON_CANONICAL_ADAPTER.ipynb")
